# 杭州租房列表爬虫（贝壳找房）

抓取贝壳找房杭州市租房列表页，提取标题、行政区、商圈、面积、月租金，并导出为 CSV。

> **说明**：贝壳有风控。请在 Cell 1 的 `COOKIES_POOL` 中填入 2~3 个浏览器 Cookie；触发登录页时会自动轮换 Cookie，全部失效时可在运行界面手动粘贴新 Cookie 后继续抓取。

In [19]:
import re
import time
import random

import pandas as pd
import requests
from lxml import etree

# ========== 可配置参数 ==========
BASE_URL = "https://hz.zu.ke.com/zufang"
MAX_PAGES = 5  # 抓取页数，可自行修改
RAW_FILE = "hangzhou_rent_raw.csv"

# Cookie 池：填入 2~3 个不同浏览器/账号的 Cookie 字符串（从 F12 → Network 复制）
# 示例: "lianjia_uuid=xxx; lianjia_ssid=xxx; ..."
COOKIES_POOL = [
    "",  # Cookie 1
    "",  # Cookie 2
    "",  # Cookie 3
]

# 常见浏览器 User-Agent（整次会话固定使用同一个，避免频繁切换触发风控）
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:121.0) Gecko/20100101 Firefox/121.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36 Edg/120.0.0.0",
]
SESSION_UA = random.choice(USER_AGENTS)


def get_active_cookies() -> list[str]:
    """返回 Cookie 池中所有非空 Cookie。"""
    return [c.strip() for c in COOKIES_POOL if isinstance(c, str) and c.strip()]

In [20]:
def build_page_url(page: int) -> str:
    if page == 1:
        return f"{BASE_URL}/"
    return f"{BASE_URL}/pg{page}/"


def build_headers(referer: str | None = None) -> dict:
    return {
        "User-Agent": SESSION_UA,
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "zh-CN,zh;q=0.9,en;q=0.8",
        "Connection": "keep-alive",
        "Referer": referer or f"{BASE_URL}/",
        "Upgrade-Insecure-Requests": "1",
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "same-origin",
    }


def apply_cookie(session: requests.Session, cookie: str | None) -> None:
    """为 Session 设置或清除 Cookie。"""
    if cookie:
        session.headers.update({"Cookie": cookie})
    else:
        session.headers.pop("Cookie", None)


def is_login_page(html: str) -> bool:
    return (
        'ke-passport" content="LOGIN"' in html
        or "<title>登录</title>" in html
        or len(html) < 5000 and "passport" in html
    )


def parse_listing_item(item) -> dict | None:
    title_nodes = item.xpath(".//p[contains(@class,'content__list--item--title')]//a/text()")
    if not title_nodes:
        return None

    title = title_nodes[0].strip()
    district_nodes = item.xpath(".//p[contains(@class,'content__list--item--des')]/a[1]/text()")
    biz_circle_nodes = item.xpath(".//p[contains(@class,'content__list--item--des')]/a[2]/text()")
    district = district_nodes[0].strip() if district_nodes else ""
    biz_circle = biz_circle_nodes[0].strip() if biz_circle_nodes else ""

    des_text = "".join(item.xpath(".//p[contains(@class,'content__list--item--des')]//text()"))
    area_match = re.search(r"([\d.]+)\s*㎡", des_text)
    area = area_match.group(1) if area_match else ""

    price_nodes = item.xpath(".//span[contains(@class,'content__list--item-price')]//em/text()")
    price = price_nodes[0].strip() if price_nodes else ""

    return {
        "title": title,
        "district": district,
        "biz_circle": biz_circle,
        "area": area,
        "price": price,
    }


def fetch_page(
    session: requests.Session,
    page: int,
    referer: str | None = None,
    cookies_pool: list[str] | None = None,
    per_cookie_retries: int = 2,
) -> str | None:
    """请求单页 HTML。

    触发风控（登录页）时不抛异常，自动轮换 Cookie 重试。
    若 Cookie 池全部失效，返回 None，由上层逻辑提示用户手动输入新 Cookie。
    """
    url = build_page_url(page)
    pool = cookies_pool if cookies_pool is not None else get_active_cookies()
    tried_cookies: set[str] = set()

    # 无 Cookie 时也尝试一次裸请求（通常仅第 1 页可能成功）
    candidates = pool.copy() if pool else [""]

    while True:
        remaining = [c for c in candidates if c not in tried_cookies]
        if not remaining:
            return None

        cookie = random.choice(remaining)
        tried_cookies.add(cookie)
        apply_cookie(session, cookie or None)

        for attempt in range(1, per_cookie_retries + 1):
            response = session.get(url, headers=build_headers(referer), timeout=20)
            response.raise_for_status()
            response.encoding = response.apparent_encoding or "utf-8"
            html = response.text

            if not is_login_page(html):
                return html

            wait = random.uniform(2.0, 4.0) * attempt
            cookie_label = f"Cookie#{len(tried_cookies)}" if cookie else "无Cookie"
            print(
                f"  -> 第 {page} 页触发风控（{cookie_label}），"
                f"{wait:.1f} 秒后重试 ({attempt}/{per_cookie_retries})"
            )
            time.sleep(wait)

        if len(tried_cookies) < len(candidates):
            print(f"  -> 当前 Cookie 失效，随机切换 Cookie 池中的其他 Cookie ...")
            time.sleep(random.uniform(1.0, 2.0))


def parse_page(html: str) -> list[dict]:
    tree = etree.HTML(html)
    items = tree.xpath("//div[@class='content__list--item']")
    records = []
    for item in items:
        record = parse_listing_item(item)
        if record:
            records.append(record)
    return records


def save_raw_records(records: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(records, columns=["title", "district", "biz_circle", "area", "price"])
    df.to_csv(RAW_FILE, index=False, encoding="utf-8-sig")
    return df

In [21]:
from IPython.display import clear_output

session = requests.Session()
session.headers.update(build_headers())

active_cookies = get_active_cookies()
if active_cookies:
    apply_cookie(session, random.choice(active_cookies))

# 先访问首页，建立会话
session.get(BASE_URL + "/", headers=build_headers(), timeout=20)
time.sleep(random.uniform(1.5, 2.5))

all_records: list[dict] = []
last_url = f"{BASE_URL}/"
page = 1

while page <= MAX_PAGES:
    print(f"正在抓取第 {page}/{MAX_PAGES} 页 ...")

    html = fetch_page(session, page, referer=last_url, cookies_pool=get_active_cookies())

    # Cookie 池全部失效时，挂起等待用户手动粘贴新 Cookie
    while html is None:
        clear_output(wait=True)
        print("=" * 55)
        print("【Cookie 全部失效，程序已暂停】")
        print(f"已成功抓取 {len(all_records)} 条，待继续第 {page}/{MAX_PAGES} 页")
        print("请浏览器打开 https://hz.zu.ke.com/zufang/ 复制 Cookie 后粘贴到下方")
        print("=" * 55)
        new_cookie = input("请粘贴新的 Cookie >>> ").strip()

        if new_cookie:
            if new_cookie not in COOKIES_POOL:
                COOKIES_POOL.append(new_cookie)
            apply_cookie(session, new_cookie)
            print(f"已接收新 Cookie，继续抓取第 {page} 页 ...")
            time.sleep(1.5)
            html = fetch_page(session, page, referer=last_url, cookies_pool=get_active_cookies())
        else:
            print("未输入 Cookie，请重新粘贴。")

    page_records = parse_page(html)
    print(f"  -> 本页解析到 {len(page_records)} 条房源")
    all_records.extend(page_records)

    # 每页成功后立即落盘，避免中途报错丢数据
    save_raw_records(all_records)
    last_url = build_page_url(page)

    if page < MAX_PAGES:
        sleep_seconds = random.uniform(3.0, 6.0)
        print(f"  -> 休眠 {sleep_seconds:.2f} 秒")
        time.sleep(sleep_seconds)

    page += 1

print(f"\n共抓取 {len(all_records)} 条记录")
if all_records:
    print(f"已自动保存至 {RAW_FILE}")
else:
    print("未抓到数据，后续单元格将尝试读取已有 CSV 文件")

正在抓取第 1/5 页 ...
  -> 本页解析到 30 条房源
  -> 休眠 4.33 秒
正在抓取第 2/5 页 ...
  -> 第 2 页触发风控，3.9 秒后重试 (1/3)
  -> 第 2 页触发风控，8.6 秒后重试 (2/3)
  -> 第 2 页触发风控，13.8 秒后重试 (3/3)
  !! 第 2 页被风控拦截，停止翻页（已保留前 30 条）
  提示：在 Cell 1 填入 COOKIE 后，重新运行 Cell 1→2→3→4

共抓取 30 条记录
已自动保存至 hangzhou_rent_raw.csv


In [22]:
import os

if all_records:
    df = save_raw_records(all_records)
elif os.path.exists(RAW_FILE):
    df = pd.read_csv(RAW_FILE, encoding="utf-8-sig")
    print(f"使用已有文件: {RAW_FILE}（{len(df)} 行）")
else:
    raise FileNotFoundError("没有抓取到数据，也没有找到 hangzhou_rent_raw.csv")

print(f"数据已保存至: {RAW_FILE}")
print("\n前 5 行预览:")
print(df.head())

数据已保存至: hangzhou_rent_raw.csv

前 5 行预览:
                                             title district biz_circle  \
0                                   整租·云河大厦 1室0厅 南      拱墅区        信义坊   
1                                  整租·保亿丽景山 3室2厅 南      余杭区      未来科技城   
2                                 整租·江山云樾北府 3室2厅 南      钱塘区       大学城北   
3  独栋·中海友里公寓 萌宠社区 民用水电/可养宠/网易600米/云狐科技园朝南一室一厅 1室1厅                       
4                                    整租·望林府 3室2厅 南      拱墅区       万达广场   

     area price  
0   51.00  2300  
1  130.00  3200  
2   88.89  3400  
3   50.00  5300  
4   98.03  4900  


# 数据预处理

读取 `hangzhou_rent_raw.csv`，完成缺失值/重复值处理、类型转换、IQR 异常值剔除、**title 特征工程**（租赁方式/卧室数/朝向及独热编码），输出 `hangzhou_rent_clean.csv`。

In [23]:
import re

INPUT_FILE = "hangzhou_rent_raw.csv"
OUTPUT_FILE = "hangzhou_rent_clean.csv"


def strip_numeric(value) -> float:
    """剥离中文字符及单位，提取数值并转为 float。"""
    if pd.isna(value):
        return float("nan")
    text = str(value).strip()
    # 仅保留数字与小数点
    numeric_text = re.sub(r"[^\d.]", "", text)
    if not numeric_text:
        return float("nan")
    return float(numeric_text)


def remove_iqr_outliers(df: pd.DataFrame, column: str) -> pd.DataFrame:
    """使用四分位距法（IQR）剔除指定列的异常值。"""
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]


# 1. 读取原始数据
df_raw = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
rows_before = len(df_raw)
print(f"清洗前行数: {rows_before}")
print(df_raw.head())

清洗前行数: 30
                                             title district biz_circle  \
0                                   整租·云河大厦 1室0厅 南      拱墅区        信义坊   
1                                  整租·保亿丽景山 3室2厅 南      余杭区      未来科技城   
2                                 整租·江山云樾北府 3室2厅 南      钱塘区       大学城北   
3  独栋·中海友里公寓 萌宠社区 民用水电/可养宠/网易600米/云狐科技园朝南一室一厅 1室1厅      NaN        NaN   
4                                    整租·望林府 3室2厅 南      拱墅区       万达广场   

     area price  
0   51.00  2300  
1  130.00  3200  
2   88.89  3400  
3   50.00  5300  
4   98.03  4900  


In [24]:
# 2. 去除缺失值与完全重复行
df_clean = df_raw.dropna().drop_duplicates().copy()
print(f"去缺失/重复后行数: {len(df_clean)} (剔除 {rows_before - len(df_clean)} 行)")

# 3. 数据类型转换：剥离 area、price 中的中文及单位，转为 float
df_clean["area"] = df_clean["area"].apply(strip_numeric)
df_clean["price"] = df_clean["price"].apply(strip_numeric)

# 转换后若产生 NaN，再次剔除
df_clean = df_clean.dropna(subset=["area", "price"])
df_clean = df_clean[(df_clean["area"] > 0) & (df_clean["price"] > 0)]
print(f"类型转换后行数: {len(df_clean)}")

# 4. 计算衍生指标：每平米单价
df_clean["unit_price"] = (df_clean["price"] / df_clean["area"]).round(2)

去缺失/重复后行数: 17 (剔除 13 行)
类型转换后行数: 17


In [25]:
# 5. IQR 异常值处理（分别对 area、price 列检测并剔除）
rows_before_iqr = len(df_clean)

df_clean = remove_iqr_outliers(df_clean, "area")
print(f"area IQR 过滤后行数: {len(df_clean)} (剔除 {rows_before_iqr - len(df_clean)} 行)")

rows_before_price_iqr = len(df_clean)
df_clean = remove_iqr_outliers(df_clean, "price")
print(f"price IQR 过滤后行数: {len(df_clean)} (剔除 {rows_before_price_iqr - len(df_clean)} 行)")

rows_after = len(df_clean)


def extract_rent_type(title: str) -> str:
    """从标题提取租赁方式。"""
    text = str(title)
    for rent_type in ("整租", "合租", "独栋"):
        if rent_type in text:
            return rent_type
    return "未知"


def extract_rooms(title: str) -> int:
    """从标题提取卧室数量（X室 / X居室）。"""
    text = str(title)
    match = re.search(r"(\d+)\s*室", text) or re.search(r"(\d+)\s*居室", text)
    return int(match.group(1)) if match else 1


def extract_orientation(title: str) -> str:
    """从标题提取朝向。"""
    text = str(title)
    for direction in ("南", "北", "东", "西"):
        if direction in text:
            return direction
    return "未知"


# 6. 特征工程：从 title 提取结构化特征
df_clean["rent_type"] = df_clean["title"].apply(extract_rent_type)
df_clean["rooms"] = df_clean["title"].apply(extract_rooms).astype(int)
df_clean["orientation"] = df_clean["title"].apply(extract_orientation)

# rent_type 独热编码（机器可读数值特征）
rent_type_dummies = pd.get_dummies(df_clean["rent_type"], prefix="rent_type", dtype=int)
df_clean = pd.concat([df_clean, rent_type_dummies], axis=1)

print("\n特征工程完成，新增列:", ["rent_type", "rooms", "orientation", *rent_type_dummies.columns.tolist()])


# 7. 输出对比与保存
print("\n========== 清洗前后行数对比 ==========")
print(f"清洗前: {rows_before} 行")
print(f"清洗后: {rows_after} 行")
print(f"共剔除: {rows_before - rows_after} 行")

df_clean.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"\n干净数据已保存至: {OUTPUT_FILE}")
print("\n特征提取后前 5 行预览:")
print(df_clean.head())

area IQR 过滤后行数: 17 (剔除 0 行)
price IQR 过滤后行数: 16 (剔除 1 行)

========== 清洗前后行数对比 ==========
清洗前: 30 行
清洗后: 16 行
共剔除: 14 行

干净数据已保存至: hangzhou_rent_clean.csv

清洗后前 5 行预览:
              title district biz_circle    area   price  unit_price
0    整租·云河大厦 1室0厅 南      拱墅区        信义坊   51.00  2300.0       45.10
1   整租·保亿丽景山 3室2厅 南      余杭区      未来科技城  130.00  3200.0       24.62
2  整租·江山云樾北府 3室2厅 南      钱塘区       大学城北   88.89  3400.0       38.25
4     整租·望林府 3室2厅 南      拱墅区       万达广场   98.03  4900.0       49.98
6    整租·广仁小区 2室1厅 南      萧山区       南部卧城   85.33  2200.0       25.78


# 统计分析与可视化

基于 `hangzhou_rent_clean.csv`，对各行政区租金水平进行分组统计，并绘制箱线图与回归散点图。

In [26]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", font="SimHei")

CLEAN_FILE = "hangzhou_rent_clean.csv"
df_analysis = pd.read_csv(CLEAN_FILE, encoding="utf-8-sig")

print(f"读取数据: {len(df_analysis)} 行")
if len(df_analysis) < 10:
    print("⚠ 数据量较少，图表效果可能不明显。建议填入 Cookie 后重新抓取。")
df_analysis.head()

读取数据: 16 行


,title,district,biz_circle,area,price,unit_price
0,整租·云河大厦 1室0厅 南,拱墅区,信义坊,51.00,2300.0,45.10
1,整租·保亿丽景山 3室2厅 南,余杭区,未来科技城,130.00,3200.0,24.62
2,整租·江山云樾北府 3室2厅 南,钱塘区,大学城北,88.89,3400.0,38.25
3,整租·望林府 3室2厅 南,拱墅区,万达广场,98.03,4900.0,49.98
4,整租·广仁小区 2室1厅 南,萧山区,南部卧城,85.33,2200.0,25.78


In [27]:
# 各行政区平均租金、平均面积、平均单价
district_stats = (
    df_analysis.groupby("district", as_index=False)
    .agg(
        平均租金=("price", "mean"),
        平均面积=("area", "mean"),
        平均单价=("unit_price", "mean"),
    )
    .round(2)
    .sort_values("平均租金", ascending=False)
)

print("杭州各行政区租房指标统计:")
print(district_stats.to_string(index=False))

杭州各行政区租房指标统计:
district   平均租金   平均面积  平均单价
     西湖区 3650.0  85.00 43.79
     拱墅区 3600.0  74.52 47.54
     上城区 3550.0  69.50 60.36
     钱塘区 3450.0  88.94 38.79
     余杭区 2700.0 105.00 30.43
     萧山区 2575.0  70.18 46.92
     临安区 1800.0 125.00 14.40


In [ ]:
# 图表 1：各行政区租金分布箱线图
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_analysis, x="district", y="price", palette="Set2")
plt.title("杭州各行政区租金分布对比", fontsize=14)
plt.xlabel("行政区")
plt.ylabel("租金（元/月）")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# 图表 2：面积与租金回归散点图
plt.figure(figsize=(10, 6))
sns.regplot(
    data=df_analysis,
    x="area",
    y="price",
    scatter_kws={"alpha": 0.6, "s": 40},
    line_kws={"color": "red", "linewidth": 2},
)
plt.title("杭州租房面积与租金相关性分析", fontsize=14)
plt.xlabel("面积（㎡）")
plt.ylabel("租金（元/月）")
plt.tight_layout()
plt.show()

# KMeans 聚类分析

基于 `hangzhou_rent_clean.csv`，结合面积、租金、单价及特征工程字段进行聚类；通过轮廓系数动态选 K，并用 PCA 降维至 3 维可视化。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

# 基础特征 + 特征工程数值列
BASE_FEATURE_COLS = ["area", "price", "unit_price", "rooms"]

df_cluster = pd.read_csv("hangzhou_rent_clean.csv", encoding="utf-8-sig")
rent_type_dummy_cols = [c for c in df_cluster.columns if c.startswith("rent_type_")]
FEATURE_COLS = BASE_FEATURE_COLS + rent_type_dummy_cols

X = df_cluster[FEATURE_COLS].copy()

# 标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA 降维至 3 个主成分（用于可视化）
pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)
explained_ratio = pca.explained_variance_ratio_

print(f"聚类特征: {FEATURE_COLS}")
print(f"样本数量: {len(df_cluster)}")
print(
    "PCA 解释方差比例: "
    f"PC1={explained_ratio[0]:.2%}, "
    f"PC2={explained_ratio[1]:.2%}, "
    f"PC3={explained_ratio[2]:.2%}, "
    f"累计={explained_ratio.sum():.2%}"
)

In [ ]:
# 遍历 K 值，计算轮廓系数并选择最优 K
# 最高 K 不超过 sqrt(样本数)，避免小样本过拟合
max_k = int(np.sqrt(len(df_cluster)))
upper_k = min(max_k, len(df_cluster) - 1)
K_RANGE = range(2, upper_k + 1)

print(f"样本数={len(df_cluster)}, sqrt(n)={max_k}, 实际搜索 K 范围: 2 ~ {upper_k}")

if len(K_RANGE) < 1:
    raise ValueError(
        f"样本量过少（{len(df_cluster)} 条），无法开展 KMeans 聚类，请先抓取更多数据。"
    )

silhouette_results = []
for k in K_RANGE:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_results.append({"K": k, "silhouette_score": round(score, 4)})
    print(f"K = {k}, 轮廓系数 = {score:.4f}")

silhouette_df = pd.DataFrame(silhouette_results)
best_k = int(silhouette_df.loc[silhouette_df["silhouette_score"].idxmax(), "K"])
print(f"\n最优 K 值: {best_k}")

In [ ]:
# 使用最优 K 值执行最终聚类（基于标准化后的高维特征）
final_kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_cluster["cluster"] = final_kmeans.fit_predict(X_scaled)

# 保存 PCA 坐标，供可视化使用
df_cluster["PC1"] = X_pca[:, 0]
df_cluster["PC2"] = X_pca[:, 1]
df_cluster["PC3"] = X_pca[:, 2]

# 各类群在核心特征上的均值
profile_cols = ["area", "price", "unit_price", "rooms"] + rent_type_dummy_cols
cluster_profile = (
    df_cluster.groupby("cluster")[profile_cols]
    .mean()
    .round(2)
    .rename(columns={
        "area": "平均面积",
        "price": "平均租金",
        "unit_price": "平均单价",
        "rooms": "平均卧室数",
    })
)

print("各聚类簇特征均值:")
print(cluster_profile)
print("\n带聚类标签的数据预览:")
print(df_cluster.head())

In [ ]:
# PCA 三维散点图：按聚类簇着色
import matplotlib.patches as mpatches

fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection="3d")

colors = sns.color_palette("Set1", n_colors=best_k)
cluster_ids = sorted(df_cluster["cluster"].unique())

for cluster_id, color in zip(cluster_ids, colors):
    subset = df_cluster[df_cluster["cluster"] == cluster_id]
    ax.scatter(
        subset["PC1"],
        subset["PC2"],
        subset["PC3"],
        c=[color],
        label=f"簇 {cluster_id}",
        s=60,
        alpha=0.85,
    )

ax.set_title("杭州租房 KMeans 聚类 PCA 三维可视化", fontsize=14)
ax.set_xlabel(f"PC1（解释方差 {explained_ratio[0]:.2%}）")
ax.set_ylabel(f"PC2（解释方差 {explained_ratio[1]:.2%}）")
ax.set_zlabel(f"PC3（解释方差 {explained_ratio[2]:.2%}）")

# 图例：聚类簇 + 各主成分解释方差比例
cluster_handles, _ = ax.get_legend_handles_labels()
pca_handles = [
    mpatches.Patch(color="none", label=f"PC1 解释方差: {explained_ratio[0]:.2%}"),
    mpatches.Patch(color="none", label=f"PC2 解释方差: {explained_ratio[1]:.2%}"),
    mpatches.Patch(color="none", label=f"PC3 解释方差: {explained_ratio[2]:.2%}"),
    mpatches.Patch(color="none", label=f"累计解释方差: {explained_ratio.sum():.2%}"),
]
ax.legend(handles=cluster_handles + pca_handles, title="聚类簇 / 主成分方差", loc="upper left")

plt.tight_layout()
plt.show()

# 空间可视化：杭州市租金单价热力地图

基于 `hangzhou_rent_clean.csv`，按行政区聚合平均租金单价，使用 pyecharts 绘制杭州市域热力地图。

In [ ]:
from pyecharts import options as opts
from pyecharts.charts import Map
from pyecharts.globals import CurrentConfig

# 使用在线地图资源（若本地未安装城市地图包，可自动加载）
CurrentConfig.ONLINE = True

MAP_OUTPUT_FILE = "hangzhou_rent_map.html"


def normalize_district(name: str) -> str:
    """规范行政区名称，确保带有「区/县/市」后缀以匹配 pyecharts 地图。"""
    name = str(name).strip()
    if name.endswith(("区", "县", "市")):
        return name
    return f"{name}区"


# 读取数据并按行政区计算平均租金单价
df_map = pd.read_csv("hangzhou_rent_clean.csv", encoding="utf-8-sig")
df_map["district_norm"] = df_map["district"].apply(normalize_district)

district_unit_price = (
    df_map.groupby("district_norm", as_index=False)["unit_price"]
    .mean()
    .round(2)
    .rename(columns={"district_norm": "district", "unit_price": "avg_unit_price"})
)

print("各行政区平均租金单价（元/㎡）:")
print(district_unit_price.to_string(index=False))

map_data = list(
    zip(
        district_unit_price["district"].tolist(),
        district_unit_price["avg_unit_price"].tolist(),
    )
)
min_price = float(district_unit_price["avg_unit_price"].min())
max_price = float(district_unit_price["avg_unit_price"].max())

In [ ]:
# 绘制杭州市域热力地图并保存为 HTML
hangzhou_map = (
    Map(init_opts=opts.InitOpts(width="1200px", height="800px", page_title="杭州租房单价地图"))
    .add(
        series_name="平均租金单价",
        data_pair=map_data,
        maptype="杭州",
        is_map_symbol_show=False,
        label_opts=opts.LabelOpts(is_show=True),
    )
    .set_global_opts(
        title_opts=opts.TitleOpts(
            title="杭州市各行政区租金单价热力图",
            subtitle="颜色越深，平均租金单价越高（元/㎡）",
            pos_left="center",
        ),
        visualmap_opts=opts.VisualMapOpts(
            min_=min_price,
            max_=max_price,
            is_calculable=True,
            range_text=["高", "低"],
            pos_left="left",
            pos_bottom="10%",
        ),
        tooltip_opts=opts.TooltipOpts(
            trigger="item",
            formatter="{b}<br/>平均单价: {c} 元/㎡",
        ),
    )
)

hangzhou_map.render(MAP_OUTPUT_FILE)
print(f"地图已保存至: {MAP_OUTPUT_FILE}")
hangzhou_map